# 01. Исследовательский анализ данных (EDA)

Кредитный скоринг: знакомство с признаками заёмщиков для **Казахстана** и **России**.

В ноутбуке: описательная статистика, пропуски, распределения, корреляции и первые выводы для главы диплома.

In [ ]:
# Пути: если запускаете из папки notebooks/, поднимаемся на уровень вверх
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import utils  # noqa: E402

utils.ensure_directories()
kz_path, ru_path = utils.ensure_raw_datasets_exist()
print("KZ:", kz_path)
print("RU:", ru_path)

In [ ]:
df_kz = pd.read_csv(kz_path, encoding="utf-8")
df_ru = pd.read_csv(ru_path, encoding="utf-8")

df_kz["country"] = "KZ"
df_ru["country"] = "RU"

print("Размер KZ:", df_kz.shape)
print("Размер RU:", df_ru.shape)
display(df_kz.head())

### Описание признаков (для текста диплома)

| Признак | Смысл |
|--------|--------|
| `age` | возраст заёмщика |
| `gender` | пол (M/F) |
| `region` | регион проживания |
| `monthly_income` | ежемесячный доход |
| `employment_status` | занятость (наём, ИП, безработный и т.д.) |
| `work_experience` | стаж (лет) |
| `loan_amount` | сумма кредита |
| `loan_term` | срок в месяцах |
| `interest_rate` | процентная ставка, % годовых |
| `current_debt` | текущая долговая нагрузка |
| `debt_to_income` | отношение долга к доходу |
| `number_of_loans` | число действующих кредитов |
| `overdue_count` | число случаев просрочки |
| `max_days_overdue` | макс. длительность просрочки (дней) |
| `loan_purpose` | цель кредита |
| `target` | **1** — дефолт, **0** — нет |

In [ ]:
for name, df in [("Казахстан", df_kz), ("Россия", df_ru)]:
    print(f"\n=== {name} ===")
    print(df.describe(include="all").T.head(20))
    print("Пропуски:\n", df.isna().sum()[df.isna().sum() > 0])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (title, df) in zip(axes, [("KZ", df_kz), ("RU", df_ru)]):
    vc = df["target"].value_counts(normalize=True).sort_index()
    ax.bar(["0 (нет)", "1 (дефолт)"], vc.values, color=["#4c72b0", "#dd8452"])
    ax.set_title(f"Доля классов — {title}")
    ax.set_ylabel("Доля")
plt.tight_layout()
plt.show()

In [ ]:
num_cols = [
    "age",
    "monthly_income",
    "loan_amount",
    "debt_to_income",
    "interest_rate",
    "overdue_count",
]
fig, axes = plt.subplots(2, 3, figsize=(12, 7))
axes = axes.ravel()
for i, col in enumerate(num_cols):
    sns.histplot(df_kz[col], kde=True, ax=axes[i], color="steelblue", label="KZ", stat="density")
    sns.histplot(df_ru[col], kde=True, ax=axes[i], color="coral", label="RU", stat="density", alpha=0.4)
    axes[i].set_title(col)
    axes[i].legend()
plt.tight_layout()
plt.show()

In [ ]:
# корреляции (пример для одной страны — для отчёта можно продублировать для RU)
plt.figure(figsize=(10, 8))
sns.heatmap(df_kz.select_dtypes(include=[np.number]).corr(), cmap="RdBu_r", center=0)
plt.title("KZ: корреляции числовых признаков")
plt.tight_layout()
plt.show()

**Краткие выводы для главы 1–2:** классы несбалансированы (~70/30), доходы и суммы кредита по странам отличаются по масштабу (валюта/рынок), при этом структура признаков совпадает — это удобно для сравнения моделей.